### 1.1 パスの設定

In [2]:
import torch
import os

# GPU設定
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ==========================================
# パス設定 
# ==========================================
DATA_DIR = "/home/nakamuraroi/kumagai/work/dataset/" 
# 読み込むファイル名
CSV_NAME = "kumagai_patentdata2.csv"
# 出力先ディレクトリ
OUTPUT_DIR = "/home/nakamuraroi/kumagai/work/processed_graph/"

os.makedirs(OUTPUT_DIR, exist_ok=True)

Using device: cuda


### 1.2 データ（../dataset/kumagai_patentdata2.csv）をロードする

In [ ]:
import pandas as pd
import numpy as np
import os
import json
import warnings
from sklearn.preprocessing import normalize
import umap
import hdbscan
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity

# 警告を無視
warnings.filterwarnings('ignore')

# ==========================================
# 設定 (環境に合わせて変更してください)
# ==========================================
DATA_DIR = "/home/nakamuraroi/kumagai/work/dataset/"
CSV_NAME = "kumagai_patentdata2.csv"

# ==========================================
# 関数定義
# ==========================================
def parse_vec(s, dim=1024):
    
    if not isinstance(s, str):
        return np.zeros(dim)
    
    # 1. ブラケットと改行を除去
    s_clean = s.replace('[', '').replace(']', '').replace('\n', ' ')
    
    # 2. カンマがある場合はJSONとして試す（念のため）
    if ',' in s_clean:
        try:
            return np.fromstring(s_clean, sep=',')
        except:
            pass
            
    # 3. スペース区切りとして変換 (高速)
    try:
        return np.fromstring(s_clean, sep=' ')
    except Exception as e:
        # どうしても無理な場合はゼロ埋め
        return np.zeros(dim)
    
def load_process_and_cluster():
    csv_path = os.path.join(DATA_DIR, CSV_NAME)
    print(f"データ読み込み中: {csv_path}")
    
    if not os.path.exists(csv_path):
        print(f"エラー: ファイルが見つかりません {csv_path}")
        return None, None

    # CSV読み込み
    df = pd.read_csv(csv_path)
    
    if 'year' in df.columns and 'month' in df.columns:
        print("year列とmonth列から日付を作成します...")
        df['year_month'] = pd.to_datetime({
            'year': df['year'],
            'month': df['month'],
            'day': 1
        })
    elif 'year_month' in df.columns:
        df['year_month'] = pd.to_datetime(df['year_month'], format='%Y-%m', errors='coerce')
    else:
        print("エラー: 日付情報(year/month または year_month)が見つかりません")
        return None, None

    # フィルタリング
    start_date = '2000-01-01'
    end_date = '2023-12-31'
    
    # 日付フィルタと筆頭発明者フィルタ
    mask = (df['year_month'] >= start_date) & (df['year_month'] <= end_date) & (df['lead_inventor'].notnull())
    df = df[mask].copy()

    print(f"有効データ数: {len(df)}件")
    
    # プログレスバーの初期化
    tqdm.pandas()

    # === ベクトル処理 ===
    print("ベクトルのパースと結合を開始...")
    
    # 1. Description Embedding (E5)
    if 'description_embedding' in df.columns:
        desc_vecs = np.stack(df['description_embedding'].progress_apply(
            lambda x: parse_vec(x, dim=1024)
        ).values)
    else:
        print("エラー: description_embedding がありません")
        return None, None

    # 2. Metadata Embedding (Node2Vec)
    if 'metadata_embedding' in df.columns:
        meta_vecs = np.stack(df['metadata_embedding'].progress_apply(
            lambda x: parse_vec(x, dim=64)
        ).values)
    else:
        print("エラー: metadata_embedding がありません")
        return None, None
    
    # 3. 正規化と結合
    print("ベクトルを正規化して結合中...")
    desc_vecs = normalize(desc_vecs, axis=1)
    meta_vecs = normalize(meta_vecs, axis=1)

    combined_vecs = np.hstack([desc_vecs, meta_vecs])
    print(f"結合後のベクトル次元数: {combined_vecs.shape[1]}")

    # === クラスタリング (UMAP + HDBSCAN) ===
    print("UMAPで次元削減中 (Cluster用)...")
    umap_embeddings = umap.UMAP(
        n_neighbors=15, 
        n_components=5, 
        metric='cosine', 
        random_state=42
    ).fit_transform(combined_vecs)

    print("HDBSCANでトピック抽出中...")
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=20,
        min_samples=1,
        metric='euclidean', 
        cluster_selection_method='eom',
        prediction_data=True
    )
    df['topic_id'] = clusterer.fit_predict(umap_embeddings)

    # ノイズ確認
    noise_count = len(df[df['topic_id'] == -1])
    n_topics = df['topic_id'].max() + 1
    print(f"抽出トピック数: {n_topics}")
    print(f"ノイズ除去された特許数: {noise_count} (全体の {noise_count/len(df)*100:.1f}%)")

    # === トピック重心の計算 ===
    print("トピック重心ベクトルを計算中...")
    topic_embeddings_dict = {}
    
    # ノイズを除外したトピックIDリスト
    valid_topic_id = sorted(list(set(df['topic_id']) - {-1}))
    centroid_matrix = []

    topic_labels = df['topic_id'].values
    
    for tid in tqdm(valid_topic_id):
        mask = (topic_labels == tid)
        topic_vec = np.mean(combined_vecs[mask], axis=0)
        topic_embeddings_dict[tid] = topic_vec
        centroid_matrix.append(topic_vec)

    noise_mask = (df['topic_id'] == -1)
    noise_count = noise_mask.sum()

    if noise_count > 0 and len(valid_topic_id) > 0:
        print(f"ノイズ {noise_count}件 を最寄りのトピックに割り当て中...")
        
        # ノイズデータのベクトルを取り出す
        noise_vecs = combined_vecs[noise_mask]
        
        # 各ノイズデータと、全トピック重心との類似度を計算
        # similarities shape: (n_noise, n_topics)
        similarities = cosine_similarity(noise_vecs, centroid_matrix)
        
        # 最も似ているトピックのインデックスを取得
        closest_indices = np.argmax(similarities, axis=1)
        
        # インデックスを実際のTopic IDに変換
        new_topic_id = [valid_topic_id[i] for i in closest_indices]
        
        # DataFrameを更新
        df.loc[noise_mask, 'topic_id'] = new_topic_id
        
        print("✓ ノイズ救済完了 (ノイズ 0件)")

    # ノイズ除去済みのDFを返す（※前の議論にあった「ノイズの救済」はここでは行わず、削除しています）
    df_clean = df[df['topic_id'] != -1].copy()
    
    return df_clean, topic_embeddings_dict

# ==========================================
# メイン実行部
# ==========================================
if __name__ == "__main__":
    print("処理を開始します...")
    
    # 関数の実行
    df_result, topic_embeds = load_process_and_cluster()
    
    if df_result is not None:
        print("\n=== 処理完了 ===")
        print(f"最終的なデータ件数: {len(df_result)}")
        print(f"作成されたトピック数: {len(topic_embeds)}")
        
        # 確認用: 先頭の数行を表示
        print("\nデータの先頭5行:")
        print(df_result[['patent_number', 'lead_inventor', 'topic_id']].head())

/home/nakamuraroi/.local/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


処理を開始します...
データ読み込み中: /home/nakamuraroi/kumagai/work/dataset/kumagai_patentdata2.csv
year列とmonth列から日付を作成します...
有効データ数: 42789件
ベクトルのパースと結合を開始...


100%|██████████| 42789/42789 [00:00<00:00, 179176.58it/s]


ベクトルを正規化して結合中...
結合後のベクトル次元数: 1088
UMAPで次元削減中 (Cluster用)...
HDBSCANでトピック抽出中...
抽出トピック数: 390
ノイズ除去された特許数: 15338 (全体の 35.8%)
トピック重心ベクトルを計算中...


100%|██████████| 390/390 [00:00<00:00, 5279.00it/s]

ノイズ 15338件 を最寄りのトピックに割り当て中...


✓ ノイズ救済完了 (ノイズ 0件)

=== 処理完了 ===
最終的なデータ件数: 42789
作成されたトピック数: 390

データの先頭5行:
      patent_number lead_inventor  topic_id
1778  特開2001-209692          中島　亨       262
1779  特開2001-205110         森内　裕之       370
1780  特開2001-206757         田浦　一英       283
1781  特開2001-207423         関本　恒浩       121
1782  特開2001-208409         橋本　直樹       388


In [4]:
df = pd.read_csv(os.path.join(DATA_DIR, CSV_NAME))
df['description_embedding']

0        [0.030059490352869034, 0.017750952392816544, -...
1        [0.011382696218788624, 0.004896808881312609, -...
2        [0.006317418534308672, 0.020430145785212517, -...
3        [0.013484649360179901, -0.002683836268261075, ...
4        [0.024531826376914978, 0.019554754719138145, -...
                               ...                        
44562    [0.04034804180264473, 0.017458472400903702, -0...
44563    [0.007959448732435703, 0.004999454133212566, -...
44564    [0.007658788003027439, 0.004220247268676758, -...
44565    [-0.0023702960461378098, 0.015371913090348244,...
44566    [0.034338876605033875, 0.005877140909433365, -...
Name: description_embedding, Length: 44567, dtype: object

## 1.3 最寄りの特許に割り当てたノイズは正しいトピックに所属しているのか？

In [5]:
# topic_id列にて，空白やNANがないことを確認する
nan_count = df_result['topic_id'].isnull().sum()
nan_count

0

In [6]:
import sys

print("ベクトルデータを数値に変換しています...")

def robust_parse(x):
    """どんな形式でも無理やり数値リストに変換する関数"""
    # 既にNumpy配列ならそのまま
    if isinstance(x, np.ndarray):
        return x.astype(np.float32)
    
    # PythonリストならNumpy配列へ
    if isinstance(x, list):
        return np.array(x, dtype=np.float32)
    
    # 文字列の場合の処理
    if isinstance(x, str):
        # 余計な記号を消す
        s = x.replace('[', '').replace(']', '').replace('\n', ' ')
        # カンマ区切りかスペース区切りか
        sep = ',' if ',' in s else ' '
        try:
            return np.fromstring(s, sep=sep, dtype=np.float32)
        except:
            pass
            
    # どうしても無理ならゼロベクトル (1024次元)
    return np.zeros(1024, dtype=np.float32)

# 1. 変換を実行
if 'tqdm' in sys.modules:
    from tqdm import tqdm
    tqdm.pandas()
    embeddings_series = df_result['description_embedding'].progress_apply(robust_parse)
else:
    embeddings_series = df_result['description_embedding'].apply(robust_parse)

# 2. スタックして行列にする
embeddings = np.stack(embeddings_series.values)

print(f"変換完了。")
print(f"データの型: {embeddings.dtype}") # float32 になっているはず
print(f"データの形状: {embeddings.shape}")

# 3. エラーが出ていた計算処理を再実行
print("\n類似度スコア (sim_score) を再計算中...")
df_result['sim_score'] = 0.0
valid_topics = [t for t in df_result['topic_id'].unique() if t != -1]

for tid in valid_topics:
    mask = (df_result['topic_id'] == tid)
    if mask.sum() == 0: continue

    # 数値化された embeddings を使う
    topic_patent_vecs = embeddings[mask]
    
    # 重心計算 (エラー箇所)
    desc_centroid = np.mean(topic_patent_vecs, axis=0).reshape(1, -1)
    
    # 類似度計算
    sims = cosine_similarity(topic_patent_vecs, desc_centroid).flatten()
    df_result.loc[mask, 'sim_score'] = sims

print("計算完了。")
print(df_result[['topic_id', 'sim_score']].head())

ベクトルデータを数値に変換しています...


100%|██████████| 42789/42789 [00:06<00:00, 6735.46it/s]


変換完了。
データの型: float32
データの形状: (42789, 1024)

類似度スコア (sim_score) を再計算中...
計算完了。
      topic_id  sim_score
1778       262   0.924119
1779       370   0.913798
1780       283   0.944726
1781       121   0.891756
1782       388   0.919459


In [7]:
df_result

,patent_number,patent_name,date,corporation,ipc,lead_ipc,fi,fterm,keyword,description,year,month,description_embedding,metadata_embedding,lead_inventor,inventors,year_month,topic_id,sim_score
1778,特開2001-209692,施設管理システム,200001,['清水建設株式会社'],"['G06F 17/30 (2006.01)', 'G06Q 50/00 (...",G06F 17/30 (2006.01),"['G06F 17/30 170C', 'G06F 17/60 122C', 'G0...","['5B049BB05', '5B049CC02', '5B049CC45', '5B049...","['情報', 'すべて', '有効']",施設施工者が施設管理の情報をすべてにわたりデジタル情報化し、施設所有者に有効な施設管理情報を...,2000,1,"[0.03135940060019493, -0.0030927385669201612, ...",[ 0.2919243 -0.18772289 0.21053162 0.224033...,中島 亨,"中島 亨,竹島育朗",2000-01-01,262,0.924119
1779,特開2001-205110,ドラフトチャンバ,200001,['大成建設株式会社'],[],B01L 1/00 (2006.01),[],[],"['化学物質', '作業室', '空調機', '清浄', '作業環境', '実験']",化学物質を取り扱うためのフードを備えたドラフトチャンバにおいて、作業室に空調設備を設けなくて...,2000,1,"[0.013121270574629307, -0.02321396768093109, -...",[ 0.27802473 -0.2309001 0.12717132 0.175692...,森内 裕之,森内 裕之,2000-01-01,370,0.913798
1780,特開2001-206757,コンクリート組成物及びトンネル覆工工法,200001,['西松建設株式会社'],"['E21D 11/10 (2006.01)', 'C04B 28/02 (...",E21D 11/10 (2006.01),"['E21D 11/10 D', 'E21D 11/10 Z', 'C0...","['2D055DB00', '2D055KA00', '4G012PA27', '4G012...","['吹付けコンクリート', '品質', 'コストダウン', '作業環境', 'コンクリート組...",吹付けコンクリートの品質、施工性を向上させると共に、コストダウンを達成し、さらには口内粉塵の...,2000,1,"[0.0016511422581970692, 0.016119062900543213, ...",[ 0.30306432 -0.20834202 0.12808324 0.177891...,田浦 一英,"田浦 一英,山本 康博,藤川 可",2000-01-01,283,0.944726
1781,特開2001-207423,後退パラペット型堤体の衝撃波力低減工法,200001,"['五洋建設株式会社', '中国電力株式会社']",['E02B 3/06 (2006.01)'],E02B 3/06 (2006.01),['E02B 3/06 301'],"['2D018BA11', '2D118AA11', '2D118DA01', '2D118...","['従来', '作用']",従来の後退パラペット型堤体においては、後退パラペットに作用する波力および転倒モーメントが大き...,2000,1,"[0.03772636130452156, 0.0183484498411417, -0.0...",[ 0.23017973 -0.32329643 0.08356746 0.327914...,関本 恒浩,"関本 恒浩,森屋 陽一,佐貫 宏,川俣 奨,泉 雄士,金田 時義,藤原 茂範,平岡 順...",2000-01-01,121,0.891756
1782,特開2001-208409,空調用の流路切換装置及びそれを備えた空調機,200001,"['株式会社日建設計', '新晃工業株式会社']",['F24F 13/02 (2006.01)'],F24F 13/02 (2006.01),['F24F 13/02 D'],"['3L080AA02', '3L080AA04']","['空調システム', '流路']",空調システムに使用される流路の切換装置を簡略化する,2000,1,"[0.009848109446465969, 0.027212440967559814, -...",[ 0.25761563 -0.20579714 0.19196749 0.156607...,橋本 直樹,"橋本 直樹,稲川 健",2000-01-01,388,0.919459
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44562,特開2021-101101,水路堆積物除去装置及び水路堆積物除去方法,202104,['東洋建設株式会社'],"['E02B 5/00 (2006.01)', 'B08B 1/04 (...",E03F 9/00 (2006.01),"['E02B 5/00 Z', 'B08B 1/04', 'E02B 15/0...",['2D025AA00'],"['駆動力', '堆積物', '掘削', '除去']",大きな駆動力を要することなく、堆積物を効率的に掘削して除去する,2021,4,"[0.04034804180264473, 0.017458472400903702, -0...",[ 0.34501505 -0.28883272 0.16848534 0.178671...,田中 啓之,"田中 啓之,宮原 和仁,森田 研志",2021-04-01,105,0.945454
44563,特開2021-119296,プレキャスト構造部材と経時硬化材の連結構造の構築方法,202105,"['オリエンタル白石株式会社', '株式会社ガイアート', '株式会社熊谷組', 'ジオスタ...",['E01D 19/12 (2006.01)'],E01D 19/12 (2006.01),['E01D 19/12'],"['2D059AA14', '2D059GG55']","['簡単', '作業手順', '養生期間', '施工期間', '連結構造', '構築方法']",簡単な方法により作業手順や養生期間を削減して施工期間を短縮するとともに、凹部のひび割れ耐力を...,2021,5,"[0.007959448732435703, 0.004999454133212566, -...",[ 0.24111861 -0.24822187 0.23449725 0.169212...,正司 明夫,"正司 明夫,大谷 悟司,櫻井 正之,渡邊 輝康,下中村 圭太,高松 芳徳",2021-05-01,11,0.935896
44564,特開2021-121724,構造物補強用部材及び継手構造,202105,"['東日本旅客鉄道株式会社', '清水建設株式会社']","['E04G 23/02 (2006.01)', 'E01D 22/00 (...",E04G 23/02 (2006.01),"['E04G 23/02 F', 'E01D 22/00 B', 'E0...","['2D059AA03', '2D059GG40', '2D059GG55', '2E176...","['継手部材', '加工手間', '施工手間', '大幅']",継手部材の加工手間と施工手間を大幅に軽減することを可能にする構造物補強用部材を提供する,2021,5,"[0.007658788003027439, 0.004220247268676758, -...",[ 0.26992404 -0.19907099 0.14545646 0.176773...,大郷 貴之,"大郷 貴之,伊東 典紀,久保 昌史,名倉 健二,杉橋 直行,前田 敏也,山下 裕司",2021-05-01,279,0.952788
44565,特開2021-121725,拡底部を備えた地下壁杭構造,202105,['大成建設株式会社'],"['E02D 5/30 (2006.01)', 'E02D 5/48 (...",E02D 5/30 (2006.01),"['E02D 5/30 Z', 'E02D 5/48', 'E02D 5/2...","['2D041AA01', '2D041BA22', '2D041CB03', '2D041...","['拡底部', '壁杭', '拡底部', '耐力', '支持力']",拡底部を備えた壁杭の拡底部の耐力が低下することを解消し、高支持力の地下壁杭を形成すること,2021,5,"[-0.0023702960461378098, 0.015371913090348244,

In [10]:
df_result[df_result['topic_id'] == 1]

,patent_number,patent_name,date,corporation,ipc,lead_ipc,fi,fterm,keyword,description,year,month,description_embedding,metadata_embedding,lead_inventor,inventors,year_month,topic_id,sim_score
1916,特開2001-193867,水底パイプの敷設方法,200001,['大成建設株式会社'],['F16L 1/12 (2006.01)'],B63B 35/03 (2006.01),[],[],"['経済的', '水底', '管', '敷設方法']",簡易な設備によって、経済的に水底にパイプを敷設することができる、水底パイプの敷設方法を提供す...,2000,1,"[-0.002034298377111554, 0.0027278983034193516,...",[ 0.23349671 -0.25231802 0.32635847 0.281826...,上田 耕平,上田 耕平,2000-01-01,1,0.939131
3651,特開2002-038446,防砂シートおよびその敷設方法,200007,['東亜建設工業株式会社'],"['E02B 3/12 (2006.01)', 'E02D 23/02 (...",E02B 3/12 (2006.01),"['E02B 3/12', 'E02D 23/02 Z']","['2D118AA05', '2D118AA28', '2D118BA05', '2D118...","['大水深', '護岸', '岸壁', '安定', '簡単', '経済的', '敷設方法']",大水深における護岸や岸壁等の背面に対し、防砂シートをより安定した状態で簡単に敷設する経済的な...,2000,7,"[0.023763136938214302, -0.0076948790811002254,...",[ 0.23627968 -0.27725416 0.33917353 0.264404...,瀧野 浩,瀧野 浩,2000-07-01,1,0.951616
4316,特開2002-098262,大水深管の敷設方法,200009,['清水建設株式会社'],['F16L 1/12 (2006.01)'],F16L 1/12 (2006.01),[],[],"['海水', '密度', '液体', '注入', '重量', '浮力', '必要', '敷設...",大水深管の管内に海水よりも低密度の液体を注入することで大水深管の重量を調整して、敷設施工する...,2000,9,"[-0.0022064237855374813, -0.001981830690056085...",[ 0.24209012 -0.2449117 0.244986 0.228197...,高岩 千人,高岩 千人,2000-09-01,1,0.924753
4569,特開2002-115398,デッキプレート敷設用吊り具及びデッキプレートの敷設方法,200010,['清水建設株式会社'],"['E04G 21/16 (2006.01)', 'E04B 5/40 (...",E04G 21/16 (2006.01),['E04G 21/16'],"['2E174AA01', '2E174BA01', '2E174CA03', '2E174...","['デッキプレート', '作業効率', 'デッキプレート', '敷設方法']",デッキプレートの敷設を容易にしてその労働負荷を低減し、しかも作業効率を高めることのできるデッ...,2000,10,"[0.019928576424717903, -0.018501145765185356, ...",[ 0.28137445 -0.2656641 0.3778134 0.260146...,山崎 忍,"山崎 忍,古口 光",2000-10-01,1,0.918378
5245,特開2002-220998,防水シートの敷設装置及び防水シートの敷設方法,200101,"['大成建設株式会社', '株式会社奥村組']",['E21D 11/38 (2006.01)'],E21D 11/38 (2006.01),['E21D 11/38 A'],"['2D055BA01', '2D055HA06', '2D055LA02', '2D155...","['良好', '接合', '防水シート', '敷設方法']",防水シトの垂下がりを防止でき、良好な接合を行え、且つ作業性を向上できる防水シートの敷設装置及...,2001,1,"[0.018293527886271477, -0.015942418947815895, ...",[ 0.30441368 -0.2980459 0.34830177 0.271297...,栄毅熾,"栄毅熾,芳賀由紀夫,島田哲治,原修一,松岡義治,尾崎裕之,馬場和徳,畑山栄一,福居雅也,平沢...",2001-01-01,1,0.953308
5282,特開2002-227178,軟弱地盤の覆土用シートおよびその敷設方法およびそれに使用する接合治具,200101,"['東亜建設工業株式会社', '三恵産業株式会社']",['E02D 3/00 (2006.01)'],E02D 3/00 (2006.01),['E02D 3/00 102'],"['2D043CA08', '2D043DD04']","['軟弱地盤', '全体', '軟弱地盤', 'ネット', 'シート', '直接', 'シー...",軟弱地盤上全体に水を張った状態ではなく、軟弱地盤とネットまたはシートが直接触れるような状態で...,2001,1,"[0.025791803374886513, -0.013960282318294048, ...",[ 2.49313951e-01 -2.58871138e-01 3.54094267e-...,多児 崇典,多児 崇典,2001-01-01,1,0.947659
6564,特開2002-363961,覆土用シートの敷設方法および敷設に使用するシート台船,200106,"['東亜建設工業株式会社', '三恵産業株式会社', '信幸建設株式会社']",['E02D 3/00 (2006.01)'],E02D 3/00 (2006.01),['E02D 3/00 102'],"['2D043CA04', '2D043CA20', '2D043DA09', '2D043...","['覆土', 'シート', '設備費', '経済的', '覆土', 'シート', '覆土',...",覆土用シートを敷設するための設備費用が比較的少なくてすみ、経済的に覆土用シートを敷設すること...,2001,6,"[0.03204285725951195, -0.0057513527572155, -0....",[ 0.2571419 -0.28072286 0.38015872 0.278919...,多児 崇典,多児 崇典,2001-06-01,1,0.944312
6710,特開2003-056772,自在継手およびそれを使用した管路の敷設方法,200107,['東亜建設工業株式会社'],['F16L 1/12 (2006.01)'],F16L 1/12 (2006.01),[],['3H104JC09'],"['海底', '管路', '必要', '線状', '取水', '管路', '管路', '取水...",海底への管路の敷設時には必要な曲がりを得ることができ、敷設後に自在継手ができるだけ直線状に戻...,2001,7,"[0.0014281379990279675, -0.002521525602787733,...",[ 0.2408062 -0.27413827 0.32860106 0.276654...,西川 豊,西川 豊,2001-07-01,1,0.942895
6739,特開2003-056771,自在継手およびそれを使用した管路の敷設方法,200107,['東亜建設工業株式会社'],['F16L 1/12 (2006.01)'],F16L 1/12 (2006.01),[],['3H104JC09'],"['海底', '管路', '必要', '取水', '管路', '管路', '取水', 'エネ...",海底への管路の敷設時には必要な曲がりを得ることができ、敷設後に自在継手ができるだけ曲がりが少...,2001,7,"[0.0019128387793898582, -0.001871743705123663,...",[ 0.24046975 -0.27397344 0.3286971 0.276883...,西川 豊,西川 豊,2001-07-01,1,0.944202
8345,特開2003-225630,二重遮水シートとその敷設方法,200202,"['若築建設株式会社', '東ソー・ニッケミ株式会社', '太洋興業株式会社', '株式会社...",['B09B 1/00 (2006.01)'],B09B 1/00 (2006.01),['B09B 1/00 F'],"['4D004AA46', '4D004BB02', '4D004BB03']","['軽量', '運搬', '敷設作業

In [11]:
df_result.csv_path = os.path.join(DATA_DIR, "topic_info2.csv")
df_result.to_csv(df_result.csv_path, index=False)

### トピック名をLLMでラベルを自動生成

#### トピックの数を減らしてクラスタリングしてtopic_info3に保存

In [21]:
import pandas as pd
import numpy as np
import os
import json
import warnings
from sklearn.preprocessing import normalize
import umap
import hdbscan
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity

# 警告を無視
warnings.filterwarnings('ignore')

# ==========================================
# 設定 (環境に合わせて変更してください)
# ==========================================
DATA_DIR = "/home/nakamuraroi/kumagai/work/dataset/"
CSV_NAME = "kumagai_patentdata2.csv"

# ==========================================
# 関数定義
# ==========================================
def parse_vec(s, dim=1024):
    
    if not isinstance(s, str):
        return np.zeros(dim)
    
    # 1. ブラケットと改行を除去
    s_clean = s.replace('[', '').replace(']', '').replace('\n', ' ')
    
    # 2. カンマがある場合はJSONとして試す（念のため）
    if ',' in s_clean:
        try:
            return np.fromstring(s_clean, sep=',')
        except:
            pass
            
    # 3. スペース区切りとして変換 (高速)
    try:
        return np.fromstring(s_clean, sep=' ')
    except Exception as e:
        # どうしても無理な場合はゼロ埋め
        return np.zeros(dim)
    
def load_process_and_cluster():
    csv_path = os.path.join(DATA_DIR, CSV_NAME)
    print(f"データ読み込み中: {csv_path}")
    
    if not os.path.exists(csv_path):
        print(f"エラー: ファイルが見つかりません {csv_path}")
        return None, None

    # CSV読み込み
    df = pd.read_csv(csv_path)
    
    if 'year' in df.columns and 'month' in df.columns:
        print("year列とmonth列から日付を作成します...")
        df['year_month'] = pd.to_datetime({
            'year': df['year'],
            'month': df['month'],
            'day': 1
        })
    elif 'year_month' in df.columns:
        df['year_month'] = pd.to_datetime(df['year_month'], format='%Y-%m', errors='coerce')
    else:
        print("エラー: 日付情報(year/month または year_month)が見つかりません")
        return None, None

    # フィルタリング
    start_date = '2000-01-01'
    end_date = '2023-12-31'
    
    # 日付フィルタと筆頭発明者フィルタ
    mask = (df['year_month'] >= start_date) & (df['year_month'] <= end_date) & (df['lead_inventor'].notnull())
    df = df[mask].copy()

    print(f"有効データ数: {len(df)}件")
    
    # プログレスバーの初期化
    tqdm.pandas()

    # === ベクトル処理 ===
    print("ベクトルのパースと結合を開始...")
    
    # 1. Description Embedding (E5)
    if 'description_embedding' in df.columns:
        desc_vecs = np.stack(df['description_embedding'].progress_apply(
            lambda x: parse_vec(x, dim=1024)
        ).values)
    else:
        print("エラー: description_embedding がありません")
        return None, None

    # 2. Metadata Embedding (Node2Vec)
    if 'metadata_embedding' in df.columns:
        meta_vecs = np.stack(df['metadata_embedding'].progress_apply(
            lambda x: parse_vec(x, dim=64)
        ).values)
    else:
        print("エラー: metadata_embedding がありません")
        return None, None
    
    # 3. 正規化と結合
    print("ベクトルを正規化して結合中...")
    desc_vecs = normalize(desc_vecs, axis=1)
    meta_vecs = normalize(meta_vecs, axis=1)

    combined_vecs = np.hstack([desc_vecs, meta_vecs])
    print(f"結合後のベクトル次元数: {combined_vecs.shape[1]}")

    # === クラスタリング (UMAP + HDBSCAN) ===
    print("UMAPで次元削減中 (Cluster用)...")
    umap_embeddings = umap.UMAP(
        n_neighbors=15, 
        n_components=5, 
        metric='cosine', 
        random_state=42
    ).fit_transform(combined_vecs)

    print("HDBSCANでトピック抽出中...")
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=250,
        min_samples=20,
        metric='euclidean', 
        cluster_selection_method='eom',
        prediction_data=True
    )
    df['topic_id'] = clusterer.fit_predict(umap_embeddings)

    # ノイズ確認
    noise_count = len(df[df['topic_id'] == -1])
    n_topics = df['topic_id'].max() + 1
    print(f"抽出トピック数: {n_topics}")
    print(f"ノイズ除去された特許数: {noise_count} (全体の {noise_count/len(df)*100:.1f}%)")

    # === トピック重心の計算 ===
    print("トピック重心ベクトルを計算中...")
    topic_embeddings_dict = {}
    
    # ノイズを除外したトピックIDリスト
    valid_topic_id = sorted(list(set(df['topic_id']) - {-1}))
    centroid_matrix = []

    topic_labels = df['topic_id'].values
    
    for tid in tqdm(valid_topic_id):
        mask = (topic_labels == tid)
        topic_vec = np.mean(combined_vecs[mask], axis=0)
        topic_embeddings_dict[tid] = topic_vec
        centroid_matrix.append(topic_vec)

    noise_mask = (df['topic_id'] == -1)
    noise_count = noise_mask.sum()

    if noise_count > 0 and len(valid_topic_id) > 0:
        print(f"ノイズ {noise_count}件 を最寄りのトピックに割り当て中...")
        
        # ノイズデータのベクトルを取り出す
        noise_vecs = combined_vecs[noise_mask]
        
        # 各ノイズデータと、全トピック重心との類似度を計算
        # similarities shape: (n_noise, n_topics)
        similarities = cosine_similarity(noise_vecs, centroid_matrix)
        
        # 最も似ているトピックのインデックスを取得
        closest_indices = np.argmax(similarities, axis=1)
        
        # インデックスを実際のTopic IDに変換
        new_topic_id = [valid_topic_id[i] for i in closest_indices]
        
        # DataFrameを更新
        df.loc[noise_mask, 'topic_id'] = new_topic_id
        
        print("✓ ノイズ救済完了 (ノイズ 0件)")

    # ノイズ除去済みのDFを返す（※前の議論にあった「ノイズの救済」はここでは行わず、削除しています）
    df_clean = df[df['topic_id'] != -1].copy()
    
    return df_clean, topic_embeddings_dict

# ==========================================
# メイン実行部
# ==========================================
if __name__ == "__main__":
    print("処理を開始します...")
    
    # 関数の実行
    df_result, topic_embeds = load_process_and_cluster()
    
    if df_result is not None:
        print("\n=== 処理完了 ===")
        print(f"最終的なデータ件数: {len(df_result)}")
        print(f"作成されたトピック数: {len(topic_embeds)}")
        
        # 確認用: 先頭の数行を表示
        print("\nデータの先頭5行:")
        print(df_result[['patent_number', 'lead_inventor', 'topic_id']].head())

処理を開始します...
データ読み込み中: /home/nakamuraroi/kumagai/work/dataset/kumagai_patentdata2.csv
year列とmonth列から日付を作成します...
有効データ数: 42789件
ベクトルのパースと結合を開始...


100%|██████████| 42789/42789 [00:00<00:00, 180761.62it/s]


ベクトルを正規化して結合中...
結合後のベクトル次元数: 1088
UMAPで次元削減中 (Cluster用)...
HDBSCANでトピック抽出中...
抽出トピック数: 7
ノイズ除去された特許数: 2399 (全体の 5.6%)
トピック重心ベクトルを計算中...


100%|██████████| 7/7 [00:00<00:00, 99.90it/s]

ノイズ 2399件 を最寄りのトピックに割り当て中...
✓ ノイズ救済完了 (ノイズ 0件)

=== 処理完了 ===
最終的なデータ件数: 42789
作成されたトピック数: 7

データの先頭5行:
      patent_number lead_inventor  topic_id
1778  特開2001-209692          中島　亨         6
1779  特開2001-205110         森内　裕之         6
1780  特開2001-206757         田浦　一英         6
1781  特開2001-207423         関本　恒浩         6
1782  特開2001-208409         橋本　直樹         6


### BisectingKMeansでトピック分類

In [30]:
import pandas as pd
import numpy as np
import os
import json
import warnings
from sklearn.preprocessing import normalize
import umap
import hdbscan
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity

# 警告を無視
warnings.filterwarnings('ignore')

# ==========================================
# 設定 (環境に合わせて変更してください)
# ==========================================
DATA_DIR = "/home/nakamuraroi/kumagai/work/dataset/"
CSV_NAME = "kumagai_patentdata2.csv"

# ==========================================
# 関数定義
# ==========================================
def parse_vec(s, dim=1024):
    
    if not isinstance(s, str):
        return np.zeros(dim)
    
    # 1. ブラケットと改行を除去
    s_clean = s.replace('[', '').replace(']', '').replace('\n', ' ')
    
    # 2. カンマがある場合はJSONとして試す（念のため）
    if ',' in s_clean:
        try:
            return np.fromstring(s_clean, sep=',')
        except:
            pass
            
    # 3. スペース区切りとして変換 (高速)
    try:
        return np.fromstring(s_clean, sep=' ')
    except Exception as e:
        # どうしても無理な場合はゼロ埋め
        return np.zeros(dim)
    
def load_process_and_cluster():
    csv_path = os.path.join(DATA_DIR, CSV_NAME)
    print(f"データ読み込み中: {csv_path}")
    
    if not os.path.exists(csv_path):
        print(f"エラー: ファイルが見つかりません {csv_path}")
        return None, None

    # CSV読み込み
    df = pd.read_csv(csv_path)
    
    if 'year' in df.columns and 'month' in df.columns:
        print("year列とmonth列から日付を作成します...")
        df['year_month'] = pd.to_datetime({
            'year': df['year'],
            'month': df['month'],
            'day': 1
        })
    elif 'year_month' in df.columns:
        df['year_month'] = pd.to_datetime(df['year_month'], format='%Y-%m', errors='coerce')
    else:
        print("エラー: 日付情報(year/month または year_month)が見つかりません")
        return None, None

    # フィルタリング
    start_date = '2000-01-01'
    end_date = '2023-12-31'
    
    # 日付フィルタと筆頭発明者フィルタ
    mask = (df['year_month'] >= start_date) & (df['year_month'] <= end_date) & (df['lead_inventor'].notnull())
    df = df[mask].copy()

    print(f"有効データ数: {len(df)}件")
    
    # プログレスバーの初期化
    tqdm.pandas()

    # === ベクトル処理 ===
    print("ベクトルのパースと結合を開始...")
    
    # 1. Description Embedding (E5)
    if 'description_embedding' in df.columns:
        desc_vecs = np.stack(df['description_embedding'].progress_apply(
            lambda x: parse_vec(x, dim=1024)
        ).values)
    else:
        print("エラー: description_embedding がありません")
        return None, None

    # 2. Metadata Embedding (Node2Vec)
    if 'metadata_embedding' in df.columns:
        meta_vecs = np.stack(df['metadata_embedding'].progress_apply(
            lambda x: parse_vec(x, dim=64)
        ).values)
    else:
        print("エラー: metadata_embedding がありません")
        return None, None
    
    # 3. 正規化と結合
    print("ベクトルを正規化して結合中...")
    desc_vecs = normalize(desc_vecs, axis=1)
    meta_vecs = normalize(meta_vecs, axis=1)

    combined_vecs = np.hstack([desc_vecs, meta_vecs])
    print(f"結合後のベクトル次元数: {combined_vecs.shape[1]}")

    # === クラスタリング (UMAP + HDBSCAN) ===
    print("UMAPで次元削減中 (Cluster用)...")
    umap_embeddings = umap.UMAP(
        n_neighbors=15, 
        n_components=5, 
        metric='cosine', 
        random_state=42
    ).fit_transform(combined_vecs)

    from sklearn.cluster import BisectingKMeans # 追加インポートが必要ならファイルの先頭へ

    TARGET_TOPIC_NUM = 10 # 目標トピック数
    print(f"Bisecting K-Meansで {TARGET_TOPIC_NUM} 個のトピックに分割中 (バランス重視)...")
    
    # BisectingKMeansの実行
    clusterer = BisectingKMeans(
        n_clusters=TARGET_TOPIC_NUM, 
        init='k-means++',
        random_state=42,
        bisecting_strategy='largest_cluster' # 最も大きいクラスタを優先的に分割
    )
    df['topic_id'] = clusterer.fit_predict(umap_embeddings)

    # K-Means系はノイズ(-1)が出ないので、ノイズ確認や救済処理はすべて削除・スキップ可能です
    print("クラスタリング完了")

    # === トピック重心の計算 ===
    print("トピック重心ベクトルを計算中...")
    topic_embeddings_dict = {}
    
    unique_topics = sorted(df['topic_id'].unique())
    topic_labels = df['topic_id'].values
    
    # 重心計算
    for tid in tqdm(unique_topics):
        mask = (topic_labels == tid)
        topic_vec = np.mean(combined_vecs[mask], axis=0)
        topic_embeddings_dict[tid] = topic_vec

    # 最終データフレーム (ノイズがないのでそのままコピー)
    df_clean = df.copy()
    
    # ノイズを除外したトピックIDリスト
    valid_topic_id = sorted(list(set(df['topic_id']) - {-1}))
    centroid_matrix = []

    topic_labels = df['topic_id'].values
    
    final_topic_counts = df_clean['topic_id'].value_counts()
    print("\nトピックごとのデータ件数（Bisecting K-Means結果）:")
    print(final_topic_counts)
    
    return df_clean, topic_embeddings_dict

# ==========================================
# メイン実行部
# ==========================================
if __name__ == "__main__":
    print("処理を開始します...")
    
    # 関数の実行
    df_result, topic_embeds = load_process_and_cluster()
    
    if df_result is not None:
        print("\n=== 処理完了 ===")
        print(f"最終的なデータ件数: {len(df_result)}")
        print(f"作成されたトピック数: {len(topic_embeds)}")
        
        # 確認用: 先頭の数行を表示
        print("\nデータの先頭5行:")
        print(df_result[['patent_number', 'lead_inventor', 'topic_id']].head())

処理を開始します...
データ読み込み中: /home/nakamuraroi/kumagai/work/dataset/kumagai_patentdata2.csv
year列とmonth列から日付を作成します...
有効データ数: 42789件
ベクトルのパースと結合を開始...


100%|██████████| 42789/42789 [00:00<00:00, 185056.56it/s]


ベクトルを正規化して結合中...
結合後のベクトル次元数: 1088
UMAPで次元削減中 (Cluster用)...
Bisecting K-Meansで 10 個のトピックに分割中 (バランス重視)...
クラスタリング完了
トピック重心ベクトルを計算中...


100%|██████████| 10/10 [00:00<00:00, 131.24it/s]


トピックごとのデータ件数（Bisecting K-Means結果）:
topic_id
5    8515
8    7858
3    7072
2    5934
1    5208
0    4000
7    1443
4     983
9     945
6     831
Name: count, dtype: int64

=== 処理完了 ===
最終的なデータ件数: 42789
作成されたトピック数: 10

データの先頭5行:
      patent_number lead_inventor  topic_id
1778  特開2001-209692          中島　亨         1
1779  特開2001-205110         森内　裕之         0
1780  特開2001-206757         田浦　一英         3
1781  特開2001-207423         関本　恒浩         5
1782  特開2001-208409         橋本　直樹         0


In [31]:
import sys

print("ベクトルデータを数値に変換しています...")

def robust_parse(x):
    """どんな形式でも無理やり数値リストに変換する関数"""
    # 既にNumpy配列ならそのまま
    if isinstance(x, np.ndarray):
        return x.astype(np.float32)
    
    # PythonリストならNumpy配列へ
    if isinstance(x, list):
        return np.array(x, dtype=np.float32)
    
    # 文字列の場合の処理
    if isinstance(x, str):
        # 余計な記号を消す
        s = x.replace('[', '').replace(']', '').replace('\n', ' ')
        # カンマ区切りかスペース区切りか
        sep = ',' if ',' in s else ' '
        try:
            return np.fromstring(s, sep=sep, dtype=np.float32)
        except:
            pass
            
    # どうしても無理ならゼロベクトル (1024次元)
    return np.zeros(1024, dtype=np.float32)

# 1. 変換を実行
if 'tqdm' in sys.modules:
    from tqdm import tqdm
    tqdm.pandas()
    embeddings_series = df_result['description_embedding'].progress_apply(robust_parse)
else:
    embeddings_series = df_result['description_embedding'].apply(robust_parse)

# 2. スタックして行列にする
embeddings = np.stack(embeddings_series.values)

print(f"変換完了。")
print(f"データの型: {embeddings.dtype}") # float32 になっているはず
print(f"データの形状: {embeddings.shape}")

# 3. エラーが出ていた計算処理を再実行
print("\n類似度スコア (sim_score) を再計算中...")
df_result['sim_score'] = 0.0
valid_topics = [t for t in df_result['topic_id'].unique() if t != -1]

for tid in valid_topics:
    mask = (df_result['topic_id'] == tid)
    if mask.sum() == 0: continue

    # 数値化された embeddings を使う
    topic_patent_vecs = embeddings[mask]
    
    # 重心計算 (エラー箇所)
    desc_centroid = np.mean(topic_patent_vecs, axis=0).reshape(1, -1)
    
    # 類似度計算
    sims = cosine_similarity(topic_patent_vecs, desc_centroid).flatten()
    df_result.loc[mask, 'sim_score'] = sims

print("計算完了。")
print(df_result[['topic_id', 'sim_score']].head())

ベクトルデータを数値に変換しています...


100%|██████████| 42789/42789 [00:06<00:00, 6860.36it/s]


変換完了。
データの型: float32
データの形状: (42789, 1024)

類似度スコア (sim_score) を再計算中...
計算完了。
      topic_id  sim_score
1778         1   0.906706
1779         0   0.909713
1780         3   0.932252
1781         5   0.882758
1782         0   0.914396


In [32]:
df_result

,patent_number,patent_name,date,corporation,ipc,lead_ipc,fi,fterm,keyword,description,year,month,description_embedding,metadata_embedding,lead_inventor,inventors,year_month,topic_id,sim_score
1778,特開2001-209692,施設管理システム,200001,['清水建設株式会社'],"['G06F 17/30 (2006.01)', 'G06Q 50/00 (...",G06F 17/30 (2006.01),"['G06F 17/30 170C', 'G06F 17/60 122C', 'G0...","['5B049BB05', '5B049CC02', '5B049CC45', '5B049...","['情報', 'すべて', '有効']",施設施工者が施設管理の情報をすべてにわたりデジタル情報化し、施設所有者に有効な施設管理情報を...,2000,1,"[0.03135940060019493, -0.0030927385669201612, ...",[ 0.2919243 -0.18772289 0.21053162 0.224033...,中島 亨,"中島 亨,竹島育朗",2000-01-01,1,0.906706
1779,特開2001-205110,ドラフトチャンバ,200001,['大成建設株式会社'],[],B01L 1/00 (2006.01),[],[],"['化学物質', '作業室', '空調機', '清浄', '作業環境', '実験']",化学物質を取り扱うためのフードを備えたドラフトチャンバにおいて、作業室に空調設備を設けなくて...,2000,1,"[0.013121270574629307, -0.02321396768093109, -...",[ 0.27802473 -0.2309001 0.12717132 0.175692...,森内 裕之,森内 裕之,2000-01-01,0,0.909713
1780,特開2001-206757,コンクリート組成物及びトンネル覆工工法,200001,['西松建設株式会社'],"['E21D 11/10 (2006.01)', 'C04B 28/02 (...",E21D 11/10 (2006.01),"['E21D 11/10 D', 'E21D 11/10 Z', 'C0...","['2D055DB00', '2D055KA00', '4G012PA27', '4G012...","['吹付けコンクリート', '品質', 'コストダウン', '作業環境', 'コンクリート組...",吹付けコンクリートの品質、施工性を向上させると共に、コストダウンを達成し、さらには口内粉塵の...,2000,1,"[0.0016511422581970692, 0.016119062900543213, ...",[ 0.30306432 -0.20834202 0.12808324 0.177891...,田浦 一英,"田浦 一英,山本 康博,藤川 可",2000-01-01,3,0.932252
1781,特開2001-207423,後退パラペット型堤体の衝撃波力低減工法,200001,"['五洋建設株式会社', '中国電力株式会社']",['E02B 3/06 (2006.01)'],E02B 3/06 (2006.01),['E02B 3/06 301'],"['2D018BA11', '2D118AA11', '2D118DA01', '2D118...","['従来', '作用']",従来の後退パラペット型堤体においては、後退パラペットに作用する波力および転倒モーメントが大き...,2000,1,"[0.03772636130452156, 0.0183484498411417, -0.0...",[ 0.23017973 -0.32329643 0.08356746 0.327914...,関本 恒浩,"関本 恒浩,森屋 陽一,佐貫 宏,川俣 奨,泉 雄士,金田 時義,藤原 茂範,平岡 順...",2000-01-01,5,0.882758
1782,特開2001-208409,空調用の流路切換装置及びそれを備えた空調機,200001,"['株式会社日建設計', '新晃工業株式会社']",['F24F 13/02 (2006.01)'],F24F 13/02 (2006.01),['F24F 13/02 D'],"['3L080AA02', '3L080AA04']","['空調システム', '流路']",空調システムに使用される流路の切換装置を簡略化する,2000,1,"[0.009848109446465969, 0.027212440967559814, -...",[ 0.25761563 -0.20579714 0.19196749 0.156607...,橋本 直樹,"橋本 直樹,稲川 健",2000-01-01,0,0.914396
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44562,特開2021-101101,水路堆積物除去装置及び水路堆積物除去方法,202104,['東洋建設株式会社'],"['E02B 5/00 (2006.01)', 'B08B 1/04 (...",E03F 9/00 (2006.01),"['E02B 5/00 Z', 'B08B 1/04', 'E02B 15/0...",['2D025AA00'],"['駆動力', '堆積物', '掘削', '除去']",大きな駆動力を要することなく、堆積物を効率的に掘削して除去する,2021,4,"[0.04034804180264473, 0.017458472400903702, -0...",[ 0.34501505 -0.28883272 0.16848534 0.178671...,田中 啓之,"田中 啓之,宮原 和仁,森田 研志",2021-04-01,2,0.928260
44563,特開2021-119296,プレキャスト構造部材と経時硬化材の連結構造の構築方法,202105,"['オリエンタル白石株式会社', '株式会社ガイアート', '株式会社熊谷組', 'ジオスタ...",['E01D 19/12 (2006.01)'],E01D 19/12 (2006.01),['E01D 19/12'],"['2D059AA14', '2D059GG55']","['簡単', '作業手順', '養生期間', '施工期間', '連結構造', '構築方法']",簡単な方法により作業手順や養生期間を削減して施工期間を短縮するとともに、凹部のひび割れ耐力を...,2021,5,"[0.007959448732435703, 0.004999454133212566, -...",[ 0.24111861 -0.24822187 0.23449725 0.169212...,正司 明夫,"正司 明夫,大谷 悟司,櫻井 正之,渡邊 輝康,下中村 圭太,高松 芳徳",2021-05-01,9,0.935755
44564,特開2021-121724,構造物補強用部材及び継手構造,202105,"['東日本旅客鉄道株式会社', '清水建設株式会社']","['E04G 23/02 (2006.01)', 'E01D 22/00 (...",E04G 23/02 (2006.01),"['E04G 23/02 F', 'E01D 22/00 B', 'E0...","['2D059AA03', '2D059GG40', '2D059GG55', '2E176...","['継手部材', '加工手間', '施工手間', '大幅']",継手部材の加工手間と施工手間を大幅に軽減することを可能にする構造物補強用部材を提供する,2021,5,"[0.007658788003027439, 0.004220247268676758, -...",[ 0.26992404 -0.19907099 0.14545646 0.176773...,大郷 貴之,"大郷 貴之,伊東 典紀,久保 昌史,名倉 健二,杉橋 直行,前田 敏也,山下 裕司",2021-05-01,8,0.946551
44565,特開2021-121725,拡底部を備えた地下壁杭構造,202105,['大成建設株式会社'],"['E02D 5/30 (2006.01)', 'E02D 5/48 (...",E02D 5/30 (2006.01),"['E02D 5/30 Z', 'E02D 5/48', 'E02D 5/2...","['2D041AA01', '2D041BA22', '2D041CB03', '2D041...","['拡底部', '壁杭', '拡底部', '耐力', '支持力']",拡底部を備えた壁杭の拡底部の耐力が低下することを解消し、高支持力の地下壁杭を形成すること,2021,5,"[-0.0023702960461378098, 0.015371913090348244,...",[ 0.296662

In [33]:
topic_counts = df_result['topic_id'].value_counts().sort_index()
print(topic_counts)

topic_id
0    4000
1    5208
2    5934
3    7072
4     983
5    8515
6     831
7    1443
8    7858
9     945
Name: count, dtype: int64


In [34]:
df_result.csv_path = os.path.join(DATA_DIR, "topic_info3.csv")
df_result.to_csv(df_result.csv_path, index=False)